<a href="https://colab.research.google.com/github/emilianomtz0189/credit-risk-scorecard/blob/main/notebooks/01_descarga_y_entorno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Descarga de datos y preparación del entorno

**Proyecto:** modelo de riesgo de crédito (scorecard) con el dataset *Give Me Some Credit* de Kaggle.

**Objetivo del modelo:** estimar la **PD (*Probability of Default*, probabilidad de incumplimiento)**: la probabilidad de que un cliente caiga en incumplimiento dentro de un horizonte de tiempo. En este dataset, un cliente es **"malo"** si tuvo **90 días o más de atraso en los siguientes 2 años**. Los 90 días son la definición estándar de incumplimiento en la industria (también la usa Basilea).

**En este notebook:**
1. Preparar el entorno: clonar el repositorio, fijar la semilla y registrar versiones.
2. Descargar los datos desde Kaggle.
3. Cargar los datos y validar su estructura.
4. Separar la muestra de prueba **antes** de explorar los datos.

## 1. Entorno

En Colab se hacen dos cosas al inicio de cada sesión:
- **Conectar Google Drive**, donde se guarda el dataset (no se sube a GitHub).
- **Clonar el repositorio**, para trabajar con la misma estructura de carpetas que en GitHub (`data/`, `src/`, `notebooks/`). Así las rutas relativas funcionan igual en Colab y en local.

In [1]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "credit-risk-scorecard"  # <- Nombre de tu repositorio
CARPETA_DRIVE = Path("/content/drive/MyDrive/credit-risk-scorecard")  # <- Donde guardaste cs-training.csv

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q  # Trae los últimos cambios de GitHub
    os.chdir(ruta_repo)
elif Path.cwd().name == "notebooks":
    # En local, el notebook se ejecuta desde notebooks/: subimos a la raíz
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

Mounted at /content/drive
Directorio de trabajo: /content/credit-risk-scorecard


### Librerías

Se instalan desde `requirements.txt`. Colab ya trae casi todo; cuando las versiones estén fijadas, esta celda asegura que cualquier sesión use exactamente las mismas.

In [2]:
!pip install -q -r requirements.txt

In [3]:
import numpy as np
import pandas as pd

from src import config
from src.datos import (
    cargar_datos_crudos,
    copiar_datos_crudos,
    dividir_entrenamiento_prueba,
    guardar_particiones,
    resumen_estructura,
)

config.fijar_semilla()   # SEMILLA = 42 en src/config.py
config.crear_carpetas()  # data/raw y data/processed

pd.set_option("display.float_format", "{:,.4f}".format)

### Registro de versiones

Un modelo de riesgo debe poder **replicarse exactamente**: las áreas de validación de modelos y auditoría tienen que obtener los mismos resultados. Las versiones de las librerías cambian resultados (por ejemplo, valores por omisión de los algoritmos), así que se registran y se fijan en `requirements.txt`.

Copia la salida de esta celda y pégala en `requirements.txt`.

In [4]:
import importlib.metadata as metadata
import platform

paquetes = ["numpy", "pandas", "scikit-learn", "scipy", "statsmodels",
            "matplotlib", "seaborn", "xgboost", "lightgbm"]

print(f"# Probado en Python {platform.python_version()}")
for paquete in paquetes:
    try:
        print(f"{paquete}=={metadata.version(paquete)}")
    except metadata.PackageNotFoundError:
        print(f"# {paquete}: no instalado")

# Probado en Python 3.13.16
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
scipy==1.16.3
statsmodels==0.15.0
matplotlib==3.10.0
seaborn==0.13.2
xgboost==3.4.1
lightgbm==4.6.0


## 2. Datos desde Google Drive

El archivo `cs-training.csv` se descarga una sola vez de Kaggle y se guarda en Drive. Aquí se copia a `data/raw/` para que todo el proyecto lo lea desde la misma ruta relativa. Otras formas de obtener los datos están en `data/README.md`.

In [5]:
copiar_datos_crudos(CARPETA_DRIVE)

Copiado cs-training.csv a data/raw/cs-training.csv


**¿Por qué solo usamos `cs-training.csv`?** El archivo `cs-test.csv` no trae la variable objetivo: Kaggle la guardó para calificar la competencia. Sin la respuesta real no se puede medir qué tan bien predice el modelo, así que la muestra de prueba se construye a partir de `cs-training.csv`.

## 3. Carga y validación de estructura

La función `cargar_datos_crudos` valida que estén todas las columnas esperadas y las renombra al español. El diccionario de variables está en `data/README.md`.

In [6]:
df = cargar_datos_crudos()
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

Filas: 150,000 | Columnas: 11


,malo,uso_credito_revolvente,edad,atrasos_30_59,razon_deuda,ingreso_mensual,num_creditos_abiertos,atrasos_90,num_creditos_inmobiliarios,atrasos_60_89,num_dependientes
id_cliente,,,,,,,,,,,
1,1,0.7661,45,2,0.8030,"9,120.0000",13,0,6,0,2.0000
2,0,0.9572,40,0,0.1219,"2,600.0000",4,0,0,0,1.0000
3,0,0.6582,38,1,0.0851,"3,042.0000",2,1,0,0,0.0000
4,0,0.2338,30,0,0.0360,"3,300.0000",5,0,0,0,0.0000
5,0,0.9072,49,1,0.0249,"63,588.0000",7,0,1,0,0.0000


In [7]:
# Tipo de dato, nulos y rango de cada variable.
# Es solo un primer vistazo: la limpieza se hace en el notebook 02.
resumen_estructura(df)

,tipo,nulos,pct_nulos,valores_unicos,minimo,maximo
malo,int64,0,0.0000,2,0.0000,1.0000
uso_credito_revolvente,float64,0,0.0000,125728,0.0000,"50,708.0000"
edad,int64,0,0.0000,86,0.0000,109.0000
atrasos_30_59,int64,0,0.0000,16,0.0000,98.0000
razon_deuda,float64,0,0.0000,114194,0.0000,"329,664.0000"
ingreso_mensual,float64,29731,19.8200,13594,0.0000,"3,008,750.0000"
num_creditos_abiertos,int64,0,0.0000,58,0.0000,58.0000
atrasos_90,int64,0,0.0000,19,0.0000,98.0000
num_creditos_inmobiliarios,int64,0,0.0000,28,0.0000,54.0000
atrasos_60_89,int64,0,0.0000,13,0.0000,98.0000


In [8]:
# Distribución de la variable objetivo: proporción de clientes buenos y malos
conteo = df[config.COLUMNA_OBJETIVO].value_counts().rename(index={0: "bueno", 1: "malo"})
print(conteo.to_string())
print(f"\nTasa de malos: {df[config.COLUMNA_OBJETIVO].mean():.2%}")

malo
bueno    139974
malo      10026

Tasa de malos: 6.68%


## 4. Separación de entrenamiento y prueba

La muestra de prueba se separa **antes de explorar o limpiar los datos** para evitar la **fuga de información (*data leakage*)**: todas las decisiones posteriores (cómo imputar nulos, dónde cortar los grupos de WoE, qué variables entran al modelo) se toman viendo solo la muestra de entrenamiento. Así, la muestra de prueba simula clientes nuevos que el modelo nunca vio, y sus métricas son una estimación honesta del desempeño real.

- **70 / 30:** deja suficientes clientes malos en prueba para que métricas como KS y AUC sean estables.
- **Estratificada:** ambas muestras conservan la misma tasa de malos. Con una clase minoritaria, una división al azar simple podría desbalancearlas.
- **Semilla fija:** la partición es idéntica en cada ejecución.

In [9]:
train, test = dividir_entrenamiento_prueba(df)

pd.DataFrame(
    {
        "clientes": [len(train), len(test)],
        "malos": [train[config.COLUMNA_OBJETIVO].sum(), test[config.COLUMNA_OBJETIVO].sum()],
        "tasa_malos": [train[config.COLUMNA_OBJETIVO].mean(), test[config.COLUMNA_OBJETIVO].mean()],
    },
    index=["entrenamiento", "prueba"],
)

,clientes,malos,tasa_malos
entrenamiento,105000,7018,0.0668
prueba,45000,3008,0.0668


In [10]:
guardar_particiones(train, test)
print("Guardado:", config.ARCHIVO_TRAIN.relative_to(config.RAIZ), "y",
      config.ARCHIVO_TEST.relative_to(config.RAIZ))

Guardado: data/processed/train.csv y data/processed/test.csv


## Resumen

- Entorno reproducible: rutas relativas, semilla fija (`SEMILLA = 42`) y versiones registradas.
- Datos descargados y validados; columnas renombradas al español.
- Muestra de prueba (30 %) separada y guardada antes de cualquier análisis.

**Siguiente paso:** notebook `02_exploracion_y_limpieza.ipynb`, trabajando solo con la muestra de entrenamiento.

> Los archivos de `data/` no se suben a GitHub. En una sesión nueva de Colab basta con volver a ejecutar este notebook: la semilla fija garantiza la misma partición.